# Module A2 - Product Category Classifier Training

This notebook trains a Deep Learning model to classify images of retail products.

It is configured to run either locally (using synthetic fallbacks) or on Kaggle with GPU enabled using a subset of the real RPC dataset.

In [ ]:
import os
import glob
import json
import shutil
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import tensorflow as tf
from tensorflow.keras import layers, models

print("TensorFlow Version:", tf.__version__)

In [ ]:
# Auto-detect Kaggle environment
KAGGLING = os.path.exists('/kaggle')

if KAGGLING:
    print("Running in Kaggle environment!")
    # Case-insensitive scan for RPC dataset
    input_dirs = []
    if os.path.exists('/kaggle/input'):
        for d in os.listdir('/kaggle/input'):
            d_lower = d.lower()
            if 'rpc' in d_lower or 'retail' in d_lower or 'product' in d_lower:
                input_dirs.append(os.path.join('/kaggle/input', d))
                
    if input_dirs:
        RPC_DATASET_DIR = input_dirs[0]
    else:
        RPC_DATASET_DIR = "/kaggle/input/rpc-a-large-scale-retail-product-checkout-dataset"
    print(f"Kaggle RPC Dataset Directory: {RPC_DATASET_DIR}")
    
    # Outputs and temporary directories on Kaggle
    DATA_DIR = "/kaggle/working/products"
    MODEL_DIR = "/kaggle/working/models"
else:
    print("Running locally.")
    RPC_DATASET_DIR = None
    DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'data', 'products'))
    MODEL_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'app', 'models'))

IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 5
CLASSES = ['shoes', 'bags', 'electronics', 'clothing', 'groceries'] # Defaults for local

# Set up GPU with mixed precision for T4 acceleration if available
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU(s) detected: {gpus}")
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy('mixed_float16')
    print("Using mixed precision float16 for fast training.")
else:
    print("No GPU detected, using CPU.")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
# Define fallback synthetic generator for local run
def create_synthetic_image(category, file_path):
    bg_color = tuple(np.random.randint(180, 255, size=3))
    img = Image.new('RGB', IMAGE_SIZE, bg_color)
    draw = ImageDraw.Draw(img)
    color = tuple(np.random.randint(20, 150, size=3))
    
    if category == 'shoes':
        draw.ellipse([20, 60, 100, 100], fill=color)
        draw.rectangle([60, 40, 100, 80], fill=color)
    elif category == 'bags':
        draw.rectangle([30, 45, 98, 95], fill=color, outline=(0, 0, 0), width=2)
        draw.arc([45, 20, 83, 45], start=180, end=0, fill=(0, 0, 0), width=3)
    elif category == 'electronics':
        draw.rectangle([30, 30, 98, 98], fill=color)
        draw.rectangle([38, 38, 90, 90], fill=(255, 255, 255))
        draw.ellipse([60, 91, 68, 97], fill=(0, 0, 0))
    elif category == 'clothing':
        draw.polygon([(64, 30), (20, 55), (35, 95), (64, 80), (93, 95), (108, 55)], fill=color)
    elif category == 'groceries':
        draw.ellipse([35, 45, 80, 90], fill=color)
        draw.ellipse([55, 45, 93, 90], fill=color)
        draw.line([64, 25, 64, 45], fill=(139, 69, 19), width=3)
        
    img.save(file_path)

# Populate working directory
if KAGGLING and os.path.exists(RPC_DATASET_DIR):
    print("Loading Kaggle RPC dataset subset via symlinks...")
    train_json_files = glob.glob(os.path.join(RPC_DATASET_DIR, "*train*.json"))
    val_json_files = glob.glob(os.path.join(RPC_DATASET_DIR, "*val*.json"))
    
    if train_json_files and val_json_files:
        with open(train_json_files[0], 'r') as f:
            train_meta = json.load(f)
        with open(val_json_files[0], 'r') as f:
            val_meta = json.load(f)
            
        cat_id_to_name = {c['id']: c['name'] for c in train_meta['categories']}
        cat_counts = Counter([ann['category_id'] for ann in train_meta['annotations']])
        top_cats = [cat_id for cat_id, count in cat_counts.most_common(5)]
        CLASSES = [cat_id_to_name[cat_id] for cat_id in top_cats]
        print("Top 5 categories parsed:", CLASSES)
        
        train_img_map = {img['id']: img['file_name'] for img in train_meta['images']}
        val_img_map = {img['id']: img['file_name'] for img in val_meta['images']}
        
        train_cat_images = {name: [] for name in CLASSES}
        for ann in train_meta['annotations']:
            cat_name = cat_id_to_name.get(ann['category_id'])
            if cat_name in train_cat_images:
                img_file = train_img_map.get(ann['image_id'])
                if img_file:
                    train_cat_images[cat_name].append(img_file)
                    
        val_cat_images = {name: [] for name in CLASSES}
        for ann in val_meta['annotations']:
            cat_name = cat_id_to_name.get(ann['category_id'])
            if cat_name in val_cat_images:
                img_file = val_img_map.get(ann['image_id'])
                if img_file:
                    val_cat_images[cat_name].append(img_file)
                    
        os.makedirs(os.path.join(DATA_DIR, 'train'), exist_ok=True)
        os.makedirs(os.path.join(DATA_DIR, 'validation'), exist_ok=True)
        
        train_img_folder = os.path.join(RPC_DATASET_DIR, "train2019")
        val_img_folder = os.path.join(RPC_DATASET_DIR, "val2019")
        
        for cat in CLASSES:
            train_cat_dir = os.path.join(DATA_DIR, 'train', cat)
            val_cat_dir = os.path.join(DATA_DIR, 'validation', cat)
            os.makedirs(train_cat_dir, exist_ok=True)
            os.makedirs(val_cat_dir, exist_ok=True)
            
            # symlink 500 train images
            for file_name in train_cat_images[cat][:500]:
                src_path = os.path.join(train_img_folder, file_name)
                dest_path = os.path.join(train_cat_dir, os.path.basename(file_name))
                if os.path.exists(src_path) and not os.path.exists(dest_path):
                    try:
                        os.symlink(src_path, dest_path)
                    except:
                        shutil.copy(src_path, dest_path)
                        
            # symlink 100 validation images
            for file_name in val_cat_images[cat][:100]:
                src_path = os.path.join(val_img_folder, file_name)
                dest_path = os.path.join(val_cat_dir, os.path.basename(file_name))
                if os.path.exists(src_path) and not os.path.exists(dest_path):
                    try:
                        os.symlink(src_path, dest_path)
                    except:
                        shutil.copy(src_path, dest_path)
        print("RPC subset successfully linked!")
else:
    print("Using local/mock configuration...")
    has_data = True
    for subset in ['train', 'validation']:
        for cls in CLASSES:
            path = os.path.join(DATA_DIR, subset, cls)
            if not os.path.exists(path) or len(os.listdir(path)) == 0:
                has_data = False
                break
    if not has_data:
        print("Generating local synthetic fallback images...")
        for subset, count in [('train', 150), ('validation', 40)]:
            for cls in CLASSES:
                path = os.path.join(DATA_DIR, subset, cls)
                os.makedirs(path, exist_ok=True)
                for i in range(count):
                    file_path = os.path.join(path, f"{cls}_{i}.jpg")
                    create_synthetic_image(cls, file_path)
        print("Synthetic data populated.")
    else:
        print("Found existing local dataset.")

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, 'train'),
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, 'validation'),
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False
)

class_names = train_ds.class_names
print("Class labels loaded:", class_names)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

# Build Model Pipeline using MobileNetV2 base
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))
x = layers.RandomFlip("horizontal")(inputs)
x = layers.RandomRotation(0.1)(x)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(CLASSES), activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)
model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

In [ ]:
model_path = os.path.join(MODEL_DIR, 'product_classifier.h5')
model.save(model_path)
print(f"Product classifier model successfully saved to: {model_path}")